# Desafío por equipos · NovaRetail
## ¿Qué grupo de clientes abandona más, y qué podemos afirmar?

**El caso.** NovaRetail es una empresa ficticia que vende en tienda, web, app y marketplace. Quiere saber **qué clientes tienen más riesgo de abandonar** (dejar de comprar en los 90 días siguientes). Antes de afirmar nada, hay que revisar los datos con cuidado.

**Qué tienen que lograr.**

| Entregable | Qué es |
| --- | --- |
| 1. Una **tabla** | Abandono por canal y segmento, con cuántos clientes hay en cada grupo |
| 2. Un **gráfico** | Un mapa de calor de esa tabla |
| 3. **Dos variables nuevas** | Con una justificación de negocio cada una |
| 4. Un **hallazgo** | Una frase, con su límite: qué *no* pueden afirmar |
| 5. Una **ficha de evidencia** | El resumen de su trabajo |

**Cómo trabajan.** En equipo, con cuatro roles que pueden rotar: **conduce** (ordena el trabajo), **implementa** (escribe el código), **valida** (controla los resultados) y **presenta** (cuenta lo que hicieron). Cada celda se corre con `Shift` + `Enter`.

Mientras trabajan, voy a pasar por los equipos preguntando siempre lo mismo. Conviene tener las respuestas listas:
1. ¿Qué decisión habilita esto?
2. ¿Cuál es la población y el período?
3. ¿Cómo evitaron fuga o sesgo?
4. ¿Qué validaron?
5. ¿Qué no pueden afirmar?

> **Dos palabras:** un **DataFrame** es una tabla (filas y columnas), y una **semilla** es el número que fija un sorteo para que todos obtengamos los mismos datos.

## Antes de empezar

Este archivo es **autocontenido**: no necesita ningún otro archivo ni datos para descargar. Usa solo tres librerías: **NumPy** (cuentas y sorteos), **pandas** (tablas) y **matplotlib** (gráficos).

In [ ]:
# Si te falta alguna librería, sacá el «#» de la línea de abajo y corré la celda una vez:
# %pip install numpy pandas matplotlib

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

print("pandas:", pd.__version__, "· numpy:", np.__version__)

## Parte 1 · Cómo se arma la tabla de clientes

Los datos son **sintéticos**: los generamos con código, así todos trabajamos con exactamente los mismos. Esta tabla se llama **`clientes_crudos`**: una fila por cliente, **tal como llegaría de un sistema real**, con algunos errores sembrados a propósito. Vamos a armarla en cinco pasos.

> **Un aviso:** algunas celdas de esta parte aparecen **con el código colapsado**. Corren igual: alcanza con ejecutarlas. Definen cómo se sortean los datos, y conocer esa regla de antemano le quita sentido al desafío. Si después querés ver cómo se generaron, **expandí el código una vez que hayan presentado**.

### 1.1 · La semilla y los parámetros

Casi todo lo que sigue es **un sorteo** (edades, canales…). Para que el sorteo sea siempre el mismo usamos una **semilla**: con la misma semilla, cualquiera que corra este código obtiene exactamente los mismos datos.

In [ ]:
SEMILLA = 2025                                 # el «número de sorteo»
rng = np.random.default_rng(SEMILLA)           # el generador de sorteos
N_CLIENTES = 8000
FECHA_CORTE = pd.Timestamp("2025-12-31")       # el «hoy» de nuestro caso
print("Listo: vamos a generar", N_CLIENTES, "clientes")

### 1.2 · Quiénes son los clientes

Sorteamos para cada cliente su **canal principal**, su **segmento** (Esencial, Plus o Premium), su **región**, su **edad** y cuándo se dio de alta.

In [ ]:
canal = rng.choice(["web", "app", "tienda"], size=N_CLIENTES, p=[0.40, 0.25, 0.35])
p_seg = {"web": [0.62, 0.28, 0.10], "app": [0.30, 0.45, 0.25], "tienda": [0.28, 0.42, 0.30]}
segmento = np.array([rng.choice(["Esencial", "Plus", "Premium"], p=p_seg[c]) for c in canal])
region = rng.choice(["Centro", "Norte", "Sur", "Litoral", "Cuyo"], size=N_CLIENTES, p=[0.30, 0.20, 0.18, 0.17, 0.15])
edad = np.clip(rng.normal(41, 12, N_CLIENTES), 18, 80).round().astype(int)
alta = pd.Timestamp("2021-01-01") + pd.to_timedelta(rng.integers(0, 1730, N_CLIENTES), unit="D")
antig = ((FECHA_CORTE - alta).days / 30.4).round().astype(int)       # meses desde el alta
print("canal, segmento, región, edad y alta: listos")

### 1.3 · Cómo compra cada cliente

Su comportamiento: **cuántas compras** hizo en el último año, **cuánto gasta por compra** (`ticket`), **hace cuántos días** compró por última vez, cuántas veces **llamó a soporte**, si usa la app, qué **descuento** recibe y qué tan **satisfecho** está.

In [ ]:
base_compras = np.select([segmento == "Esencial", segmento == "Plus"], [5, 9], 14)
compras = rng.poisson(base_compras)
ticket = np.round(np.select([segmento == "Esencial", segmento == "Plus"], [35, 62], 110) * rng.lognormal(0, 0.25, N_CLIENTES), 2)
dias = np.clip(rng.gamma(2.2, 24, N_CLIENTES), 1, 365).round().astype(int)
soporte = rng.poisson(np.where(segmento == "Esencial", 1.2, 0.6))
usa_app = np.where(canal == "app", 1, (rng.random(N_CLIENTES) < 0.35).astype(int))
descuento = np.clip(rng.normal(np.where(segmento == "Esencial", 0.14, 0.08), 0.05, N_CLIENTES), 0, 0.35).round(3)
sat_real = np.clip(np.round(rng.normal(3.7 - 0.12 * soporte + 0.15 * (segmento == "Premium"), 0.9, N_CLIENTES)), 1, 5)
print("compras, ticket, recencia, soporte, app, descuento y satisfacción: listos")

### 1.4 · Quién abandona

Esta es la variable que queremos entender: **`abandono_90d`** (1 = el cliente se fue en los 90 días siguientes al corte). Se genera con una regla interna que **no vamos a mostrar**: descubrir qué la explica es justamente el desafío.

In [ ]:
z_dias = (dias - 60) / 40
logit = (-2.45
         + 0.85 * (segmento == "Esencial") - 0.45 * (segmento == "Premium")
         + 0.80 * z_dias + 0.30 * soporte - 0.42 * (sat_real - 3.7)
         - 0.50 * usa_app - 0.012 * (antig - 24) - 0.04 * (compras - 8)
         + 0.55 * ((canal == "tienda") & (segmento == "Esencial")))
prob_abandono = 1 / (1 + np.exp(-logit))
abandono = (rng.random(N_CLIENTES) < prob_abandono).astype(int)
print(f"Abandonan {abandono.mean():.1%} de los clientes")

### 1.5 · Armamos la tabla `clientes_crudos` (con defectos a propósito)

Como en una base real, **no todo viene perfecto**. Rompemos algunas cosas a propósito:

- A algunos clientes **les falta** la satisfacción, el ticket o la región. La satisfacción falta más entre quienes abandonan: **el faltante dice algo**.
- Algunas **edades son imposibles** (0, −1, 150).
- Agregamos `motivo_baja`, que **solo existe para quienes ya abandonaron**. Es una *fuga*: una variable que usa información del futuro.
- Duplicamos **60 filas**, como por un error de carga.

In [ ]:
sat = sat_real.copy()
sat[rng.random(N_CLIENTES) < np.where(abandono == 1, 0.30, 0.09)] = np.nan   # la satisfacción falta más entre quienes se van
ticket = ticket.astype(float)
ticket[rng.random(N_CLIENTES) < 0.03] = np.nan
region_s = pd.Series(region, dtype=object)
region_s[rng.random(N_CLIENTES) < 0.02] = np.nan
edad = edad.astype(float)
malos = rng.random(N_CLIENTES) < 0.004
edad[malos] = rng.choice([0, 150, -1], size=malos.sum())                      # edades imposibles
motivo = np.where(abandono == 1, rng.choice(["precio", "servicio", "competencia", "mudanza"], size=N_CLIENTES), None)

clientes_crudos = pd.DataFrame({
    "id_cliente": [f"NR{i:06d}" for i in range(1, N_CLIENTES + 1)],
    "fecha_alta": alta.strftime("%Y-%m-%d"), "canal": canal, "segmento": segmento, "region": region_s,
    "edad": edad, "antiguedad_meses": antig, "compras_12m": compras, "ticket_promedio": ticket,
    "dias_desde_ultima_compra": dias, "contactos_soporte_90d": soporte, "usa_app": usa_app,
    "descuento_promedio": descuento, "satisfaccion": sat, "motivo_baja": motivo, "abandono_90d": abandono,
})
repetidas = clientes_crudos.sample(60, random_state=SEMILLA)                  # 60 filas repetidas por «error de carga»
clientes_crudos = pd.concat([clientes_crudos, repetidas], ignore_index=True).sample(frac=1, random_state=SEMILLA).reset_index(drop=True)
clientes_crudos.head()

Esto es lo que significa cada columna:

In [ ]:
pd.DataFrame({
    "columna": ["id_cliente", "fecha_alta", "canal", "segmento", "region", "edad", "antiguedad_meses", "compras_12m",
                "ticket_promedio", "dias_desde_ultima_compra", "contactos_soporte_90d", "usa_app", "descuento_promedio",
                "satisfaccion", "motivo_baja", "abandono_90d"],
    "qué es": ["Identificador del cliente (debería ser único)", "Cuándo se dio de alta", "Canal principal: web, app o tienda",
               "Esencial, Plus o Premium", "Región (puede faltar)", "Edad en años (hay valores imposibles)", "Meses desde el alta",
               "Compras en los últimos 12 meses", "Importe promedio por compra (puede faltar)", "Días desde la última compra",
               "Veces que llamó a soporte en 90 días", "1 si usa la app", "Descuento promedio (de 0 a 1)",
               "Encuesta de 1 a 5 (falta más entre quienes se van)", "Por qué se fue: ¡solo existe si ya se fue!",
               "LO QUE QUEREMOS ENTENDER: 1 si abandonó en los 90 días siguientes"]})

**Qué vimos:** `clientes_crudos` tiene una fila por cliente (más 60 repetidas) y 16 columnas, con trampas sembradas.

## Parte 2 · La tabla `clientes`: los datos, ya limpios

Para empezar el desafío **desde una base confiable**, ya limpiamos la tabla. Cada arreglo es una decisión, y la dejamos escrita:

| Columna | Defecto | Decisión |
| --- | --- | --- |
| `id_cliente` | filas repetidas | quedarse con una fila por cliente |
| `edad` | valores imposibles | los que están fuera de 18–100 pasan a «faltante» |
| `region` | faltante | una categoría explícita: «Sin dato» |
| `satisfaccion` | falta más entre quienes se van | **no se rellena**: se agrega un indicador de «faltó» |
| `motivo_baja` | solo existe después de abandonar | **no se usa**: es fuga |

In [ ]:
clientes = clientes_crudos.drop_duplicates(subset="id_cliente").copy()                # un cliente, una fila
clientes["edad"] = clientes["edad"].where(clientes["edad"].between(18, 100))          # edades imposibles → faltante
clientes["region"] = clientes["region"].fillna("Sin dato")                            # región faltante → categoría explícita
clientes["satisfaccion_faltante"] = clientes["satisfaccion"].isna().astype(int)       # el faltante informa: se marca, no se rellena
# motivo_baja NO se usa como variable: solo existe después del abandono (fuga)

print(f"Clientes: {len(clientes_crudos):,} → {len(clientes):,}")
print(f"Tasa de abandono: {clientes_crudos.drop_duplicates('id_cliente')['abandono_90d'].mean():.1%} antes · {clientes['abandono_90d'].mean():.1%} después")

**Desde acá trabajan sobre la tabla `clientes`** (la limpia), **no** sobre `clientes_crudos`.

## Parte 3 · El desafío

Cada paso tiene un **ejemplo resuelto** (siempre sobre otra pregunta) y un **«tu turno»**.

### Paso A · La tabla
**Ejemplo resuelto: abandono por región.** Se agrupa por una columna y, para cada grupo, se cuenta cuántos clientes hay y qué proporción abandona. Fijense que mostramos el **tamaño del grupo**: una tasa sin tamaño engaña (un 50 % sobre 2 clientes no es lo mismo que sobre 2.000).

In [ ]:
# EJEMPLO RESUELTO: abandono por región
(clientes.groupby("region")["abandono_90d"]
         .agg(clientes="size", tasa_abandono="mean")
         .reset_index().round(3))

**Tu turno:** ahora lo mismo, pero para **canal y segmento**. Observen qué cambia entre el ejemplo de arriba y lo que piden. La tabla final tiene que tener las columnas `canal`, `segmento`, `clientes` y `tasa_abandono`.

In [ ]:
# 👉 TU TURNO: abandono por canal y segmento
tabla = None          # reemplazá None por tu código
tabla

### Paso B · El gráfico
Un **mapa de calor** muestra de un vistazo dónde está el problema: cuanto más oscuro, más abandono. Cada celda dice también el tamaño del grupo (`n`). Primero, la función que lo dibuja (correla una vez):

In [ ]:
def mapa_de_calor(tabla):
    """Mapa de calor del abandono por canal y segmento. La tabla necesita las columnas canal, segmento, clientes y tasa_abandono."""
    if not isinstance(tabla, pd.DataFrame):
        print("Primero armá la tabla del Paso A.")
        return
    faltan = [col for col in ["canal", "segmento", "clientes", "tasa_abandono"] if col not in tabla.columns]
    if faltan:
        print("Todavía faltan estas columnas en tu tabla:", faltan)
        return
    tasa = tabla.pivot(index="segmento", columns="canal", values="tasa_abandono")
    n = tabla.pivot(index="segmento", columns="canal", values="clientes")
    fig, ax = plt.subplots(figsize=(6.4, 3.4))
    im = ax.imshow(tasa.values, cmap="Purples", vmin=0, vmax=max(0.35, float(np.nanmax(tasa.values))))
    ax.set_xticks(range(len(tasa.columns)), tasa.columns)
    ax.set_yticks(range(len(tasa.index)), tasa.index)
    for i in range(tasa.shape[0]):
        for j in range(tasa.shape[1]):
            ax.text(j, i, f"{tasa.values[i, j]:.0%}\nn={int(n.values[i, j])}", ha="center", va="center",
                    color="white" if tasa.values[i, j] > 0.2 else "black", fontsize=10)
    ax.set_title("Abandono a 90 días por canal y segmento")
    fig.colorbar(im, ax=ax, label="tasa de abandono")
    plt.tight_layout()
    plt.show()

In [ ]:
mapa_de_calor(tabla)

### Paso C · Dos variables nuevas
Una **variable nueva** es una columna que construimos a partir de las que ya hay, para capturar algo del negocio. La regla: **tiene que responder a una pregunta de negocio** y **no puede usar información del futuro** (como `motivo_baja`, ni el propio `abandono_90d`).

**Ejemplo resuelto:** `ticket_x_compras` = lo que gasta por compra × cuántas compras hizo. Mide cuánto aporta un cliente en el año. *Justificación: los clientes que aportan poco podrían ser los que más fácil se van.*

In [ ]:
# EJEMPLO RESUELTO
clientes["ticket_x_compras"] = clientes["ticket_promedio"] * clientes["compras_12m"]
clientes["ticket_x_compras"].describe().round(2)

**Tu turno:** creen **dos variables más**. Miren las columnas que tiene `clientes` y pregúntense qué comportamiento de un cliente podría anticipar que se va. Escriban en el comentario **por qué** cada una importa para el negocio.

In [ ]:
# 👉 TU TURNO · variable 1
# ¿Qué pregunta de negocio responde? (escribilo acá, en un comentario)
clientes["var_1"] = None          # reemplazá None por tu cálculo, usando columnas de la tabla

In [ ]:
# 👉 TU TURNO · variable 2
# ¿Qué pregunta de negocio responde?
clientes["var_2"] = None          # reemplazá None por tu cálculo

Una mirada rápida a sus variables nuevas: ¿tienen datos?, ¿qué valores toman?

In [ ]:
clientes[["var_1", "var_2"]].describe()

### Paso D · El hallazgo
Escriban **una frase** que resuma lo que encontraron. Un buen hallazgo:

- dice **dónde**: qué grupo, con qué valor;
- da el **tamaño del grupo**: sobre cuántos clientes se apoya;
- declara su **límite**: qué **no** pueden afirmar con estos datos.

Escríbanla en la celda de abajo.

**Nuestro hallazgo:**

*(editen esta celda y escriban acá su frase)*

### Paso E · La ficha de evidencia
Al final, cada equipo resume su trabajo. Abajo hay un **ejemplo ya completado** (sobre el análisis por región): editen esta celda de texto y reemplacen cada línea por la de su equipo.

**Ficha de evidencia · Equipo: ________**

- **Decisión que habilita:** *Ejemplo: decidir en qué región conviene reforzar la retención primero.*
- **Población y período:** *Ejemplo: 8.000 clientes únicos al 31/12/2025; abandono = baja en los 90 días siguientes.*
- **Procedimiento:** *Ejemplo: se limpió la tabla (sin repetidos, edades imposibles a faltante, `motivo_baja` afuera) y se agrupó por región.*
- **Resultado:** *Ejemplo: la región con más abandono es X (Y %, con n clientes).*
- **Validación:** *Ejemplo: se comprobó que la limpieza conservó los 8.000 clientes y la tasa de abandono.*
- **Límite (qué no podemos afirmar):** *Ejemplo: describe dónde se abandona más; no prueba por qué ni qué efecto tendría una acción.*

### Si se traban
- Trabajen siempre sobre la tabla **`clientes`** (la limpia), no sobre `clientes_crudos`.
- Miren de nuevo el **ejemplo resuelto** de cada paso: está pensado como modelo.
- Una variable nueva es una operación entre columnas ya existentes.
- Controlen siempre dos cosas: **cuántos clientes** tiene cada grupo y **si el resultado cambia** al cambiar una decisión.
- Pregúntense en cada paso: ¿esto me dice **dónde** o me dice **por qué**?